In [0]:
from pyspark.sql.functions import col, round, when

In [0]:
sales = spark.table(
    "workspace.pharma_silver.silver_sales"
)

drugs = spark.table(
    "workspace.pharma_silver.silver_drugs"
)

In [0]:
df_sales_gold = (
    sales
    .join(
        drugs.select(
            "drug_id",
            "drug_name",
            "generic_name",
            "brand_name",
            "drug_category"
        ),
        on="drug_id",
        how="left"
    )
)

In [0]:
df_sales_gold = df_sales_gold.withColumn(
    "sales_revenue",
    round(
        col("quantity_sold") * col("unit_price"),
        2
    )
)

In [0]:
df_sales_gold = df_sales_gold.withColumn(
    "payment_category",
    when(
        col("payment_status") == "Paid",
        "Completed"
    )
    .when(
        col("payment_status") == "Pending",
        "Pending"
    )
    .when(
        col("payment_status") == "Overdue",
        "Outstanding"
    )
    .otherwise(
        "Cancelled"
    )
)

In [0]:
df_sales_gold = df_sales_gold.select(
    "sale_id",
    "drug_id",
    "drug_name",
    "generic_name",
    "brand_name",
    "drug_category",
    "warehouse_location",
    "sale_date",
    "quantity_sold",
    "unit_price",
    "sales_revenue",
    "customer_type",
    "payment_status",
    "payment_category"
)

In [0]:
display(df_sales_gold)

In [0]:
display(
    sales.select(
        "sale_id",
        "quantity_sold",
        "unit_price",
        "revenue"
    )
)       

In [0]:
df_sales_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "workspace.pharma_gold.gold_sales_analytics"
    )